In [ ]:
import numpy as np
import emcee
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

This is an implementation of the Voigt-Reuss-Hill model on the original probablistic inversion with mcmc.

In [67]:
# ----- Fluid End-Member Properties -----
K_water = 2.2e9      # Bulk modulus of water, Pa
rho_water = 1000.0   # Density of water, kg/m³

K_gas = 1e5          # Bulk modulus for gas, Pa
rho_gas = 1.8        # Density for gas, kg/m³

In [68]:
def voigt_bulk(K_s, K_f, phi):
    return phi * K_f + (1 - phi) * K_s

In [69]:
def reuss_bulk(K_s, K_f, phi):
    return 1.0 / (phi / K_f + (1 - phi) / K_s)

In [70]:
def hill_bulk(K_s, K_f, phi):
    K_V = voigt_bulk(K_s, K_f, phi)
    K_R = reuss_bulk(K_s, K_f, phi)
    return 0.5 * (K_V + K_R)

In [71]:
def effective_shear_modulus(G_s, phi):
    # For fluid: G_f = 0, so Voigt average is (1-phi)*G_s,
    # Reuss average is 0, so Hill average:
    return 0.5 * (1 - phi) * G_s

In [72]:
def effective_density(rho_s, rho_f, phi):
    return (1 - phi) * rho_s + phi * rho_f

In [73]:
def seismic_velocities(K_eff, G_eff, rho_eff):
    Vp = np.sqrt((K_eff + (4.0/3.0) * G_eff) / rho_eff)
    Vs = np.sqrt(G_eff / rho_eff)
    return Vp, Vs

In [ ]:
def forward(theta):
    """
    Forward model mapping unknown parameters to effective seismic properties.
    
    theta = [S_w, phi, K_s, G_s, rho_s]
      S_w: water saturation (0 to 1)
      phi: crack porosity (volume fraction)
      K_s: matrix bulk modulus (Pa)
      G_s: matrix shear modulus (Pa)
      rho_s: matrix density (kg/m³)
      
    Returns: [Vp, Vs, rho_eff]
    """
    S_w, phi, K_s, G_s, rho_s = theta
    K_f = S_w * K_water + (1 - S_w) * K_gas
    rho_f = S_w * rho_water + (1 - S_w) * rho_gas
    K_eff = hill_bulk(K_s, K_f, phi)
    G_eff = effective_shear_modulus(G_s, phi)
    rho_eff = effective_density(rho_s, rho_f, phi)
    Vp, Vs = seismic_velocities(K_eff, G_eff, rho_eff)
    return np.array([Vp, Vs, rho_eff])

In [ ]:
Vp_obs = 4700.0       # m/s (5 km/s)
Vs_obs = 2700.0       # m/s (2.8 km/s)
# For effective density, assume matrix density ~3000 kg/m³ with water-filled pores:
phi_obs = 0.01
rho_s_obs = 2589.0
rho_fluid_obs = 1000.0
rho_obs = (1 - phi_obs) * rho_s_obs + phi_obs * rho_fluid_obs  # ~2980 kg/m³
sigma_Vp = 300.0       # m/s
sigma_Vs = 100.0       # m/s
sigma_rho = 157.0      # kg/m³

In [ ]:
def log_likelihood(theta):
    model = forward(theta)  # [Vp, Vs, rho_eff]
    diff = model - np.array([Vp_obs, Vs_obs, rho_obs])
    chi2 = (diff[0]**2/sigma_Vp**2 +
            diff[1]**2/sigma_Vs**2 +
            diff[2]**2/sigma_rho**2)
    return -0.5 * chi2

In [ ]:
def log_prior(theta):
    S_w, phi, K_s, G_s, rho_s = theta
    if (0.0 < S_w < 1.0 and 
        0.001 < phi < 0.5 and 
        756e8 < K_s < 80e9 and
        256e8 < G_s < 40e9 and
        2680 < rho_s < 2900):
        return 0.0
    return -np.inf

In [ ]:
def log_post(theta):
    theta = np.asarray(theta).ravel()
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf, np.full(3, np.nan)
    ll = log_likelihood(theta)
    blob = forward(theta)  # Blob: [Vp, Vs, rho_eff]
    return lp + ll, blob

In [ ]:
lb = np.array([0.0, 0.001, 756e8, 256e8, 2680])
ub = np.array([1.0, 0.5, 80e9, 40e9, 2900])
n = np.shape(ub)[0]
H = np.array([1,1,1], dtype=int)
Ne = 3 * n
prior_pdf = np.random.uniform(lb, ub, (Ne, n))
d = np.array([Vp_obs, Vs_obs, rho_obs])
s = np.array([sigma_Vp, sigma_Vs, sigma_rho])
sampler = emcee.EnsembleSampler(Ne, n, log_post)
Nsteps = 50000
sampler.run_mcmc(prior_pdf, Nsteps, progress=True)
blobs = sampler.get_blobs()

# Extract samples and analyze results
samples = sampler.get_chain(flat=True)

# Identify best-fit sample
flat_log_prob = sampler.get_log_prob(flat=True)
best_idx = np.argmax(flat_log_prob)
best_params = samples[best_idx]
best_model = forward(best_params)

# Plot results
labels = ["Water Content", "Porosity", "Bulk Modulus", "Shear Modulus", "Density"]
fig, axes = plt.subplots(n, figsize=(10, 7), sharex=True)
for i in range(n):
    axes[i].plot(samples[:, i], "k", alpha=0.3)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel("Step Number")
plt.tight_layout()
plt.show()

In [ ]:
def plot_1d_histograms(samples, labels=None, bins=100, savefig=None):
    n_parameters = samples.shape[1]  
    if labels is None:
        labels = [f"Parameter {i+1}" for i in range(n_parameters)]
    
    fig, axes = plt.subplots(n_parameters, 1, figsize=(8, 2 * n_parameters), sharex=False)
    if n_parameters == 1:
        axes = [axes]
    
    for i in range(n_parameters):
        ax = axes[i]
        ax.hist(samples[:, i], bins=bins, density=True, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_ylabel("Density")
        ax.set_title(labels[i])
    axes[-1].set_xlabel("Parameter Value")
    
    plt.tight_layout()
    
    plt.show()
    
    if savefig:
        plt.savefig(savefig)

In [ ]:
label = np.array(['water saturation', 'porosity', 'mineral bulk modulus', 'mineral shear modulus', 'mineral density'])
plot_1d_histograms(samples, labels = label, bins=100, savefig = 'vrh_fig1.png')

In [ ]:
param_names = list(label)

# Find the indices for porosity and water saturation
idx_porosity = param_names.index('porosity')
idx_water    = param_names.index('water saturation')

# Function to plot and save a single histogram
def plot_and_save(param_idx, param_name, minn, maxx, mayy, bins=100, filename=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(samples[:, param_idx], bins=bins, density=True, alpha=0.7, edgecolor='black')
    #ax.set_xlabel(param_name.capitalize())
    #ax.set_ylabel("Density")
    #ax.set_title(f"Histogram of {param_name.capitalize()}")
    ax.set_xlim(minn, maxx)
    ax.tick_params(
    axis='both',         # apply to both x & y axes
    which='major',       # only affect major ticks
    labelsize=20,        # font size of tick labels
    length=8,            # length of tick marks in points
    width=1.5            # width of the tick marks
    )
    ax.set_ylim(0, mayy)
    plt.tight_layout()
    if filename:
        fig.savefig(filename)
    plt.show()

np.save("saturation_vrh.npy", np.asarray(idx_water))
np.save("porosity_vrh.npy", np.asarray(idx_porosity))
# Porosity
plot_and_save(idx_porosity, 'crack porosity', 0, 0.5, 13, bins=100, filename='porosity_vrh_5.png')

# Water saturation
plot_and_save(idx_water, 'water saturation', 0, 1.0, 6, bins=100, filename='saturation_vrh_5.png')

In [ ]:
bb1 = blobs.reshape(-1, 3)
mask = np.all(np.isfinite(bb1), axis=1)
bb_good = bb1[mask]
post_lab = ['vp', 'vs', 'rhob']
plot_1d_histograms(bb_good, labels=post_lab, bins=100, savefig = "vrh_fig2.png")

In [ ]:
def plot_2d_color_plots(samples, lab, savefig=None):
    """
    Plot 2D color maps (histograms) for selected pairs of MCMC parameters.
    
    Parameters:
      samples: numpy array of shape (N, ndim) with MCMC chain samples.
      lab:     list of parameter labels (length ndim).
      savefig: (Optional) Filename to save the figure (e.g., "my_2dplots.png").
    """
    # Extract individual variables from the samples.
    water_saturation = samples[:, 0]
    porosity = samples[:, 1]
    mineral_bulk_modulus = samples[:, 2]
    mineral_shear_modulus = samples[:, 3]
    mineral_density = samples[:, 4]

    # List of variables to plot.
    variables = [water_saturation, porosity, mineral_bulk_modulus,
                 mineral_shear_modulus, mineral_density]

    # Define a list of pair indices to plot.
    pairs = [
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 2), (1, 3), (2, 4),
        (0, 3), (1, 4),
        (0, 4), (1, 0)
    ]
    total_pairs = len(pairs)

    # Create a grid for the subplots.
    nrows = 4
    ncols = 3
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 15))
    
    plot_counter = 0
    for i in range(nrows):
        for j in range(ncols):
            if plot_counter < total_pairs:
                x_idx, y_idx = pairs[plot_counter]

                # Select the pair of variables to plot.
                x_var = variables[x_idx]
                y_var = variables[y_idx]

                # Compute the 2D histogram with 100 bins per axis.
                hist, x_edges, y_edges = np.histogram2d(x_var, y_var, bins=100)

                # Plot the 2D color map using imshow.
                im = axs[i, j].imshow(hist.T, extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                                        origin="lower", aspect="auto", cmap="viridis")
                axs[i, j].set_title(f"{lab[y_idx]} vs {lab[x_idx]}")
                axs[i, j].set_xlabel(lab[x_idx])
                axs[i, j].set_ylabel(lab[y_idx])
                
                plot_counter += 1
            else:
                axs[i, j].axis('off')
    
    fig.tight_layout()
    fig.colorbar(im, ax=axs, orientation='horizontal', fraction=0.02, pad=0.04)
    
    
    plt.show()
    if savefig:
        plt.savefig(savefig)
    

# Example usage:
labels = [
    "Water Saturation", 
    "Porosity", 
    "Matrix Bulk Modulus (Pa)", 
    "Matrix Shear Modulus (Pa)", 
    "Matrix Density (kg/m\u00b3)"
]

# Assuming 'samples' is your flattened MCMC chain (numpy array) with 5 columns.
plot_2d_color_plots(samples, labels, savefig="vrh_fig3.png")

In [85]:
def W_thickness(S_w, phi):
    return 8500*S_w*phi

print("rough estimate of water layer thickness (m):", W_thickness(best_params[0], best_params[1]))

rough estimate of water layer thickness (m): 14.597086989463124


In [ ]:
def marginal_mode(x, bins=100):
    counts, edges = np.histogram(x, bins=bins)
    # find bin with max count, then take its center
    idx = np.argmax(counts)
    return 0.5*(edges[idx] + edges[idx+1])

# theta = [S_w, phi, ...], so samples[:,0] = S_w, samples[:,1] = phi
S_w_mode = marginal_mode(samples[:,0], bins=100)
phi_mode = marginal_mode(samples[:,1], bins=100)
print("Histogram-mode phi:", phi_mode)
print("Histogram-mode S_w:", S_w_mode)
print("Mode-based thickness:", W_thickness(S_w_mode, phi_mode))